#tomtom_collector.py
Production-ready tile collector for TomTom Traffic Flow Raster Tiles.

Features:
- Downloads XYZ tiles for TomTom traffic flow
- Concurrent downloads with retries and exponential backoff
- Stitching tiles into a single image
- Saves metadata JSON for each snapshot


In [ ]:
# Import requirements
!pip install apscheduler
import os
import sys
import math
import time
import json
import argparse
import logging
import hashlib
from datetime import datetime, timezone, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Tuple, List, Dict
import requests
from requests.adapters import HTTPAdapter, Retry
from PIL import Image
from apscheduler.schedulers.blocking import BlockingScheduler
from google.colab import userdata
from threading import Lock
from zoneinfo import ZoneInfo

TomTom_key = userdata.get("TomTom")
os.environ['TomTom'] = TomTom_key


In [ ]:
# ---------------------------
# Configuration / Defaults
# ---------------------------
DEFAULT_ZOOM = 14
DEFAULT_RADIUS = 6  # radius in tiles around center -> (2*radius+1)^2 tiles
DEFAULT_STYLE = "relative"  # tomtom: relative0 | absolute | relative
DEFAULT_OUTPUT_DIR = "dataset"
TILE_SIZE = 512
MAX_WORKERS = 8
RATE_LIMIT_DELAY = 0.0  # base delay between requests if needed (seconds)
REQUEST_TIMEOUT = 30  # seconds
CITY_CORDS = {
    "Riyadh": {"lat": 24.7136, "lon": 46.6753},
    "Jeddah": {"lat": 21.5294, "lon": 39.1611},
    "Dammam": {"lat": 26.4241, "lon": 50.0905},
    "Al khobar": {"lat": 26.2199, "lon": 50.1932},
    "Dhahran": {"lat": 26.2381, "lon": 50.0430},
    "Al Qatif": {"lat": 26.5781, "lon": 49.9985},
}

# ---------------------------
# Logging
# ---------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger("tomtom-collector")

In [ ]:
# ---------------------------
# Utilities
# ---------------------------

# HTTP Session with Retries
def create_session(total_retries=5, backoff_factor=0.5, status_forcelist=(429, 500, 502, 503, 504)):
    session = requests.Session()
    retries = Retry(
        total=total_retries,
        backoff_factor=backoff_factor,
        status_forcelist=status_forcelist,
        allowed_methods=["GET", "HEAD"]
    )
    adapter = HTTPAdapter(max_retries=retries)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session

SESSION = create_session()


# File helpers
def safe_mkdir(path: str):
    os.makedirs(path, exist_ok=True)


def tile_cache_path(output_dir: str, index: str, zoom: int, x: int, y: int) -> str:
    return os.path.join(output_dir, "tiles", str(zoom), f"{index}_{x}_{y}.png")


# Ttile math
def latlon_to_tile(lat_deg: float, lon_deg: float, zoom: int) -> Tuple[int, int]:
    """
    Convert latitude/longitude to XYZ tile coordinates.
    """
    lat_rad = math.radians(lat_deg)
    n = 2.0 ** zoom
    x_tile = int((lon_deg + 180.0) / 360.0 * n)
    y_tile = int((1.0 - math.log(math.tan(lat_rad) + (1 / math.cos(lat_rad))) / math.pi) / 2.0 * n)
    return x_tile, y_tile


def tile_bounds(x: int, y: int, zoom: int) -> Tuple[float, float, float, float]:
    """
    Return lat/lon bounds of the tile as (north, west, south, east)
    """
    n = 2.0 ** zoom
    lon_w = x / n * 360.0 - 180.0
    lon_e = (x + 1) / n * 360.0 - 180.0
    lat_n = math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * y / n))))
    lat_s = math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * (y + 1) / n))))
    return lat_n, lon_w, lat_s, lon_e


# TomTom tile URL builder
def tomtom_tile_url(x: int, y: int, zoom: int, api_key: str, style: str = DEFAULT_STYLE) -> str:
    # Example:
    # https://api.tomtom.com/traffic/map/4/tile/flow/relative/14/9312/5978.png?key=YOUR_KEY
    base = "https://api.tomtom.com/traffic/map/4/tile/flow"
    return f"{base}/{style}/{zoom}/{x}/{y}.png?tileSize={TILE_SIZE}&key={api_key}"


def time_str_to_hm(tstr):
    """Convert 'HH:MM' into hour, minute ints."""
    time_obj = datetime.strptime(tstr, "%H:%M")
    return time_obj.hour, time_obj.minute

In [ ]:
# ---------------------------
# Downloading tiles (concurrent)
# ---------------------------
def download_tile(session: requests.Session, url: str, dst: str, timeout: int = REQUEST_TIMEOUT) -> bool:
    """
    Downloads a single tile into dst path. Returns True on success.
    Will not re-download if file exists.
    """
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        log.debug("Cache hit: %s", dst)
        return True

    safe_mkdir(os.path.dirname(dst))
    try:
        log.debug("GET %s", url)
        r = session.get(url, timeout=timeout)
        if r.status_code == 200:
            with open(dst, "wb") as f:
                f.write(r.content)
            return True
        else:
            log.warning("Failed to download %s -> status=%s", url, r.status_code)
            return False
    except Exception as e:
        log.warning("Exception while downloading %s -> %s", url, e)
        return False

def download_tiles_for_grid(api_key: str, center_lat: float, center_lon: float, zoom: int,
                            radius: int, output_dir: str, index:str, style: str = DEFAULT_STYLE,
                            max_workers: int = MAX_WORKERS, rate_delay: float = RATE_LIMIT_DELAY) -> List[Dict]:
    """
    Download the square grid of tiles centered at lat/lon of size (2*radius+1)^2.
    Returns a list of metadata dictionaries for each tile (x,y,zoom,path).
    """
    center_x, center_y = latlon_to_tile(center_lat, center_lon, zoom)
    tasks = []
    for dx in range(-radius, radius + 1):
        for dy in range(-radius, radius + 1):
            x = center_x + dx
            y = center_y + dy
            dst = tile_cache_path(output_dir, index, zoom, x, y)
            url = tomtom_tile_url(x, y, zoom, api_key, style)
            tasks.append((x, y, url, dst))

    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        future_to_task = {}
        for (x, y, url, dst) in tasks:
            future = ex.submit(download_tile, SESSION, url, dst)
            future_to_task[future] = (x, y, dst, url)

            # optional small delay to avoid bursting requests
            if rate_delay:
                time.sleep(rate_delay)

        for fut in as_completed(future_to_task):
            x, y, dst, url = future_to_task[fut]
            ok = fut.result()
            results.append({"x": x, "y": y, "zoom": zoom, "path": dst, "url": url, "success": ok})

    return results


In [ ]:
# ---------------------------
# Stitching tiles
# ---------------------------
def stitch_tiles(output_dir: str, index: str, center_lat: float, center_lon: float, zoom: int, radius: int,
                 stitched_path: str) -> Tuple[int, int]:
    """
    Stitch cached tiles into a single image and save to stitched_path.
    Returns (width, height) in pixels.
    """
    center_x, center_y = latlon_to_tile(center_lat, center_lon, zoom)
    size = 2 * radius + 1
    full_w = size * TILE_SIZE
    full_h = size * TILE_SIZE

    # create blank canvas (RGBA to preserve transparency if any)
    full_img = Image.new("RGBA", (full_w, full_h))

    missing = 0
    for dx in range(-radius, radius + 1):
        for dy in range(-radius, radius + 1):
            x = center_x + dx
            y = center_y + dy
            tile_path = tile_cache_path(output_dir, index, zoom, x, y)
            px = (dx + radius) * TILE_SIZE
            py = (dy + radius) * TILE_SIZE
            if os.path.exists(tile_path):
                try:
                    tile = Image.open(tile_path).convert("RGBA")
                    full_img.paste(tile, (px, py))
                except Exception as e:
                    log.warning("Failed to open tile %s: %s", tile_path, e)
                    missing += 1
            else:
                log.debug("Missing tile %s", tile_path)
                missing += 1

    # convert to RGB and save
    rgb = full_img.convert("RGB")
    safe_mkdir(os.path.dirname(stitched_path))
    rgb.save(stitched_path, format="PNG")
    log.info("Stitched image saved: %s (missing tiles: %d)", stitched_path, missing)
    return full_w, full_h


In [ ]:
# ---------------------------
# Take single data instance for a city
# ---------------------------

def save_metadata(output_dir: str,index: str, metadata: Dict):
    idx_dir = os.path.join(output_dir, "metadata")
    safe_mkdir(idx_dir)
    # filename with timestamp + hash for uniqueness
    KSA_timezone = timezone(timedelta(hours=3))
    timestamp = metadata.get("timestamp", datetime.now(KSA_timezone).isoformat())
    short_ts = timestamp.replace(":", "-")
    hashid = hashlib.sha1(json.dumps(metadata, sort_keys=True).encode()).hexdigest()[:8]
    fname = os.path.join(idx_dir, f"{index}_{short_ts}.json")
    with open(fname, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    log.info("Saved metadata: %s", fname)


# Snapshot flow
def take_snapshot(api_key: str, city_name: str, index: str, lat: float, lon: float, zoom: int, radius: int, output_dir: str,
             style: str, interval_seconds: int, max_workers: int) -> Dict:
    """
    One snapshot: download tiles, stitch, save metadata, return metadata dict.
    """
    safe_mkdir(output_dir)
    KSA_timezone = timezone(timedelta(hours=3))
    ts = datetime.now(KSA_timezone).isoformat()
    log.info("Starting snapshot at %s center=(%f,%f) zoom=%d radius=%d", ts, lat, lon, zoom, radius)

    # 1) download tiles
    tile_results = download_tiles_for_grid(
        api_key=api_key,
        center_lat=lat,
        center_lon=lon,
        zoom=zoom,
        radius=radius,
        output_dir=output_dir,
        index=index,
        style=style,
        max_workers=max_workers,
    )

    # 2) stitched path
    stitched_dir = os.path.join(output_dir, "stitched")
    safe_mkdir(stitched_dir)
    stitched_fn = f"map_{city_name}_{index}_{lat:.6f}_{lon:.6f}_z{zoom}_r{radius}_{ts.replace(':', '-')}.png"
    stitched_path = os.path.join(stitched_dir, stitched_fn)

    # 3) stitch
    width, height = stitch_tiles(output_dir, index, lat, lon, zoom, radius, stitched_path)

    # 4) metadata
    metadata = {
        "city": city_name,
        "timestamp": ts,
        "center_lat": lat,
        "center_lon": lon,
        "zoom": zoom,
        "radius": radius,
        "style": style,
        "stitched_image": stitched_path,
        "stitched_size": {"width": width, "height": height},
        "tiles": tile_results,
        "tile_grid_size": (2 * radius + 1, 2 * radius + 1),
        "api": "TomTom Traffic Flow Raster",
        "note": "tile coords are XYZ.",
    }

    save_metadata(output_dir, index, metadata)
    log.info("Sleeping %d seconds until next snapshot...", interval_seconds)
    time.sleep(interval_seconds)
    return metadata


In [ ]:
# ---------------------------
# Inline
# ---------------------------
def inline_parse_args(city_name):
    """
    Returns configuration values directly from code constants.
    Modify values here as needed.
    """


    lat = CITY_CORDS[city_name]["lat"]
    lon = CITY_CORDS[city_name]["lon"]


    return {
        "api_key": os.environ.get("TomTom"),  # replace with "your-key-here"
        "lat": lat,              # set latitude here
        "lon": lon,              # set longitude here
        "city_name": city_name,
        "zoom": DEFAULT_ZOOM,
        "radius": DEFAULT_RADIUS,
        "style": DEFAULT_STYLE,
        "output_dir": os.path.join(DEFAULT_OUTPUT_DIR, city_name),
        "interval": 300,             # seconds between snapshots
        "workers": MAX_WORKERS,
        "run_once": True,            # True => capture only one snapshot
        "rate_delay": RATE_LIMIT_DELAY,
    }

def collect(city: str, index: dict, lock: any):

    with lock:
        index["value"] += 1
        this_run_index = index["value"]

    args = inline_parse_args(city)
    index_str = str(this_run_index).zfill(5)
    if not args["api_key"]:
        log.error("TomTom API key not provided. Set --api-key or TOMTOM_API_KEY env var.")
        sys.exit(2)

    log.info("Starting TomTom collector with city=%s center=(%f,%f) zoom=%d radius=%d style=%s",
             args["city_name"], args["lat"], args["lon"], args["zoom"], args["radius"], args["style"])

    # adjust session or global rate delay if provided
    global RATE_LIMIT_DELAY
    RATE_LIMIT_DELAY = args["rate_delay"]

    # run
    take_snapshot(
        api_key=args["api_key"],
        city_name=args["city_name"],
        index=index_str,
        lat=args["lat"],
        lon=args["lon"],
        zoom=args["zoom"],
        radius=args["radius"],
        output_dir=args["output_dir"],
        style=args["style"],
        interval_seconds=args["interval"],
        max_workers=args["workers"],
    )


In [ ]:

# -------------------------
# Scheduling Tables
# -------------------------

FREQ_MINUTES = 1  # universal frequency

CITIES = {
    "Riyadh": {
        "type": "continuous",
        "interval_minutes": FREQ_MINUTES,
        "index": {"value": 0}
    },

    "Jeddah": {
        "type": "windowed",
        "weekday_windows": {
            # 0=Mon ... 6=Sun
            4: [("16:00", "19:00")],  # Friday windows
        },
        "default_windows": [
            ("06:00", "08:00"),
            ("14:00", "16:00")
        ],
        "interval_minutes": FREQ_MINUTES,
        "index": {"value": 0}
    },

    "Dammam": {
        "type": "windowed",
        "weekday_windows": {
            4: [("16:00", "19:00")],  # Friday windows
        },
        "default_windows": [
            ("06:00", "08:00"),
            ("14:00", "16:00")
        ],
        "interval_minutes": FREQ_MINUTES,
        "index": {"value": 0}
    },

    "Al khobar": {
        "type": "windowed",
        "weekday_windows": {
            4: [("16:00", "19:00")],  # Friday windows
        },
        "default_windows": [
            ("06:00", "08:00"),
            ("14:00", "16:00")
        ],
        "interval_minutes": FREQ_MINUTES,
        "index": {"value": 0}
    },

    "Dhahran": {
        "type": "windowed",
        "weekday_windows": {
            4: [("16:00", "19:00")],  # Friday windows
        },
        "default_windows": [
            ("06:00", "08:00"),
            ("14:00", "16:00")
        ],
        "interval_minutes": FREQ_MINUTES,
        "index": {"value": 0}
    },

    "Al Qatif": {
        "type": "windowed",
        "weekday_windows": {
            4: [("16:00", "19:00")],  # Friday windows
        },
        "default_windows": [
            ("06:00", "08:00"),
            ("14:00", "16:00")
        ],
        "interval_minutes": FREQ_MINUTES,
        "index": {"value": 0}
    }
}


# -------------------------
# Main Scheduler Setup
# -------------------------

def schedule_continuous_jobs(scheduler, city: str, index: dict, lock: any, interval_minutes: int):


    scheduler.add_job(
        collect,
        "cron",
        args=[city, index, lock],
        minute=f"*/{interval_minutes}",
        id=f"{city}_continuous",
        max_instances=3,
        coalesce=True

    )
    logging.info(f"Scheduled continuous job for {city}")


def schedule_windowed_jobs(scheduler, city: str, index: dict, lock: any, weekday_windows: dict, default_windows, interval_minutes: str):
    for weekday in range(7):
        windows = weekday_windows.get(weekday, default_windows)
        for start_str, end_str in windows:
            h1, m1 = time_str_to_hm(start_str)
            h2, m2 = time_str_to_hm(end_str)

            scheduler.add_job(
                collect,
                "cron",
                args=[city, index, lock],
                minute=f"*/{interval_minutes}",
                hour=f"{h1}-{h2}",
                day_of_week=str(weekday),
                id=f"{city}_{weekday}_{start_str.replace(':','')}_{end_str.replace(':','')}",
                max_instances=3,
                coalesce=True

            )
            logging.info(
                f"Scheduled {city} job with interval {interval_minutes} every\
                 {start_str}-{end_str} on {weekday} (hour: {h1}-{h2})"
            )


def build_scheduler():
    riyadh_tz = ZoneInfo("Asia/Riyadh")
    scheduler = BlockingScheduler(timezone=riyadh_tz)

    for city, cfg in CITIES.items():
        CITIES[city]["lock"] = Lock()

        if cfg["type"] == "continuous":
            # Every N minutes, 24 hours/day
            schedule_continuous_jobs(scheduler, city, cfg["index"], cfg["lock"], cfg['interval_minutes'])
        else:
            schedule_windowed_jobs(scheduler, city, cfg["index"], cfg["lock"],
                                   cfg.get('weekday_windows', {}),
                                   cfg.get('default_windows', []),
                                   cfg['interval_minutes'])

    return scheduler


logging.info("Starting traffic data scheduler...")
scheduler = build_scheduler()
scheduler.start()